In [0]:
 # Databricks notebook source

# MAGIC %md
# MAGIC # Bank Fraud Detection — XGBoost Model
# MAGIC
# MAGIC Source:
# MAGIC `ujjivan_2.gold.transaction_features`
# MAGIC
# MAGIC Model:
# MAGIC `ujjivan_2.gold.bank_fraud_xgboost`
# MAGIC
# MAGIC Prediction table:
# MAGIC `ujjivan_2.gold.fraud_predictions`
# MAGIC
# MAGIC User-defined prediction S3 path:
# MAGIC `s3://ujjivanpoc/Gold/fraud_predictions/`


# COMMAND ----------
# MAGIC %md
# MAGIC ## 1. Install XGBoost
# MAGIC
# MAGIC Run this cell first.



# COMMAND ----------
# MAGIC %md
# MAGIC ## 2. Restart Python
# MAGIC
# MAGIC Run this cell after the installation completes.
# MAGIC
# MAGIC After the restart, continue from the next cell.





# COMMAND ----------
# MAGIC %md
# MAGIC ## 3. Imports


# COMMAND ----------

import pandas as pd
import mlflow
import mlflow.xgboost
import xgboost

from mlflow.models import infer_signature

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

from xgboost import XGBClassifier

from pyspark.ml.feature import StringIndexer


print("========================================")
print("Environment")
print("========================================")
print("XGBoost version:", xgboost.__version__)
print("MLflow version :", mlflow.__version__)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 4. Configuration


# COMMAND ----------

# ============================================================
# CATALOG / SCHEMA
# ============================================================

CATALOG = "ujjivan_2"
SCHEMA = "gold"


# ============================================================
# SOURCE DATA
# ============================================================

SOURCE_TABLE = "ujjivan_2.gold.transaction_features"


# ============================================================
# PREDICTION TABLE
# ============================================================

PREDICTION_TABLE = "ujjivan_2.gold.fraud_predictions"


# ============================================================
# UNITY CATALOG MODEL
# ============================================================
#
# IMPORTANT:
# This is the Unity Catalog model name.
# It is NOT an S3 path.
#

MODEL_NAME = "ujjivan_2.gold.bank_fraud_xgboost"


# ============================================================
# USER-DEFINED S3 LOCATION FOR PREDICTIONS
# ============================================================

PREDICTION_S3_PATH = "s3://ujjivanpoc/Gold/fraud_predictions/"


# ============================================================
# MODEL SETTINGS
# ============================================================

TARGET_RECALL = 0.80

RANDOM_STATE = 42


print("========================================")
print("Configuration")
print("========================================")

print("Source table       :", SOURCE_TABLE)
print("Prediction table   :", PREDICTION_TABLE)
print("Model name         :", MODEL_NAME)
print("Prediction S3 path :", PREDICTION_S3_PATH)
print("Target recall      :", TARGET_RECALL)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 5. Configure Unity Catalog Model Registry


# COMMAND ----------

mlflow.set_registry_uri("databricks-uc")

print(
    "MLflow registry URI:",
    mlflow.get_registry_uri()
)

print(
    "Unity Catalog model:",
    MODEL_NAME
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 6. Load Source Table


# COMMAND ----------

df = spark.table(SOURCE_TABLE)

print("Source table:", SOURCE_TABLE)

print("Total records:", df.count())


# COMMAND ----------
# MAGIC %md
# MAGIC ## 7. Check Fraud Distribution


# COMMAND ----------

df.groupBy("Is_Fraud") \
    .count() \
    .orderBy("Is_Fraud") \
    .show()


# COMMAND ----------
# MAGIC %md
# MAGIC ## 8. Inspect Schema


# COMMAND ----------

df.printSchema()


# COMMAND ----------
# MAGIC %md
# MAGIC ## 9. Define Categorical Columns


# COMMAND ----------

categorical_columns = [
    "Gender",
    "State",
    "City",
    "Account_Type",
    "Merchant_Category",
    "Transaction_Type",
    "Device_Type"
]

print("Categorical columns:")

for column in categorical_columns:
    print(" -", column)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 10. Validate Categorical Columns


# COMMAND ----------

missing_categorical = [
    c
    for c in categorical_columns
    if c not in df.columns
]

if missing_categorical:

    raise ValueError(
        "The following categorical columns "
        "are missing from the source table: "
        + str(missing_categorical)
    )

print(
    "All categorical columns are available."
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 11. Create Indexed Categorical Columns


# COMMAND ----------

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_columns
]


for indexer in indexers:

    df = indexer.fit(df).transform(df)


indexed_columns = [
    f"{c}_idx"
    for c in categorical_columns
]

print(
    "Indexed columns created:"
)

for column in indexed_columns:
    print(" -", column)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 12. Define Feature Columns


# COMMAND ----------

feature_columns = [

    "Age",

    "Transaction_Amount",

    "Account_Balance",

    "Hour",

    "DayOfWeek",

    "Month",

    "Weekend",

    "HighValue",

    "DigitalTransaction",

    "Previous_Amount",

    "Amount_Difference",

    "Avg_Last10",

    "Customer_Total_Transactions",

    "Merchant_Diversity",

    "Gender_idx",

    "State_idx",

    "City_idx",

    "Account_Type_idx",

    "Merchant_Category_idx",

    "Transaction_Type_idx",

    "Device_Type_idx"

]

target_column = "Is_Fraud"


print("Number of features:", len(feature_columns))

print("\nFeatures:")

for feature in feature_columns:
    print(" -", feature)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 13. Validate All Required Columns


# COMMAND ----------

required_columns = (
    feature_columns
    + [target_column]
)

missing_columns = [
    c
    for c in required_columns
    if c not in df.columns
]

if missing_columns:

    raise ValueError(
        "Missing required columns: "
        + str(missing_columns)
    )

print(
    "All required columns are available."
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 14. Prepare ML Dataset


# COMMAND ----------

ml_df = df.select(
    feature_columns + [target_column]
)

print(
    "ML dataset records:",
    ml_df.count()
)

print(
    "ML dataset columns:",
    len(ml_df.columns)
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 15. Convert to Pandas


# COMMAND ----------

pdf = ml_df.toPandas()

print(
    "Pandas dataset shape:",
    pdf.shape
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 16. Clean Missing / Infinite Values


# COMMAND ----------

pdf = pdf.replace(
    [float("inf"), float("-inf")],
    pd.NA
)

pdf = pdf.fillna(0)

print(
    "Null values remaining:",
    pdf.isnull().sum().sum()
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 17. Create X and y


# COMMAND ----------

X = pdf[feature_columns].copy()

y = pdf[target_column].copy()

print("X shape:", X.shape)

print("y shape:", y.shape)

print("\nTarget distribution:")

print(
    y.value_counts()
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 18. Validate Target


# COMMAND ----------

unique_targets = sorted(
    y.dropna().unique().tolist()
)

print(
    "Target values:",
    unique_targets
)

if not set(unique_targets).issubset({0, 1}):

    raise ValueError(
        "Is_Fraud must contain only 0 and 1. "
        f"Found: {unique_targets}"
    )


# COMMAND ----------
# MAGIC %md
# MAGIC ## 19. Train / Test Split


# COMMAND ----------

X_train_full, X_test, y_train_full, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=RANDOM_STATE,

    stratify=y

)


print("Training/Test split complete.")

print(
    "Train full:",
    X_train_full.shape
)

print(
    "Test:",
    X_test.shape
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 20. Train / Validation Split


# COMMAND ----------

X_train, X_val, y_train, y_val = train_test_split(

    X_train_full,

    y_train_full,

    test_size=0.15,

    random_state=RANDOM_STATE,

    stratify=y_train_full

)


print("Training   :", X_train.shape)

print("Validation :", X_val.shape)

print("Testing    :", X_test.shape)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 21. Calculate Class Imbalance Weight


# COMMAND ----------

negative = (
    y_train == 0
).sum()

positive = (
    y_train == 1
).sum()


if positive == 0:

    raise ValueError(
        "No fraud transactions found "
        "in training data."
    )


scale_pos_weight = (
    negative / positive
)


print(
    "Normal transactions:",
    negative
)

print(
    "Fraud transactions:",
    positive
)

print(
    "scale_pos_weight:",
    scale_pos_weight
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 22. Create XGBoost Model


# COMMAND ----------

fraud_model = XGBClassifier(

    n_estimators=300,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    objective="binary:logistic",

    eval_metric="aucpr",

    early_stopping_rounds=20,

    random_state=RANDOM_STATE,

    n_jobs=-1
)


print(fraud_model)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 23. Train XGBoost


# COMMAND ----------

fraud_model.fit(

    X_train,

    y_train,

    eval_set=[
        (X_val, y_val)
    ],

    verbose=False
)


print(
    "========================================"
)

print(
    "XGBoost training completed"
)

print(
    "========================================"
)

print(
    "Best iteration:",
    fraud_model.best_iteration
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 24. Generate Test Probabilities


# COMMAND ----------

y_probability = (
    fraud_model
    .predict_proba(X_test)[:, 1]
)


print(
    "Minimum probability:",
    y_probability.min()
)

print(
    "Maximum probability:",
    y_probability.max()
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 25. Select Threshold Based on Recall


# COMMAND ----------

precisions, recalls, thresholds = (
    precision_recall_curve(
        y_test,
        y_probability
    )
)


valid_idx = [

    i

    for i, recall in enumerate(
        recalls[:-1]
    )

    if recall >= TARGET_RECALL

]


if valid_idx:

    chosen_threshold = (
        thresholds[valid_idx[-1]]
    )

else:

    chosen_threshold = 0.5


print(
    "Target recall:",
    TARGET_RECALL
)

print(
    "Chosen threshold:",
    chosen_threshold
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 26. Generate Predictions


# COMMAND ----------

y_pred_default = (
    y_probability >= 0.5
).astype(int)


y_pred_tuned = (
    y_probability >= chosen_threshold
).astype(int)


print(
    "Default predictions generated."
)

print(
    "Tuned predictions generated."
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 27. Evaluate Default Threshold


# COMMAND ----------

print(
    "\n========================================"
)

print(
    "DEFAULT THRESHOLD = 0.50"
)

print(
    "========================================"
)


print(
    classification_report(
        y_test,
        y_pred_default
    )
)


print(
    "Confusion Matrix:"
)


print(
    confusion_matrix(
        y_test,
        y_pred_default
    )
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 28. Evaluate Tuned Threshold


# COMMAND ----------

print(
    "\n========================================"
)

print(
    f"TUNED THRESHOLD = {chosen_threshold:.6f}"
)

print(
    "========================================"
)


print(
    classification_report(
        y_test,
        y_pred_tuned
    )
)


print(
    "Confusion Matrix:"
)


print(
    confusion_matrix(
        y_test,
        y_pred_tuned
    )
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 29. Calculate ROC-AUC and PR-AUC


# COMMAND ----------

roc_auc = roc_auc_score(
    y_test,
    y_probability
)


pr_auc = average_precision_score(
    y_test,
    y_probability
)


print(
    "ROC-AUC:",
    roc_auc
)

print(
    "PR-AUC:",
    pr_auc
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 30. Prepare Prediction Data


# COMMAND ----------

prediction_pdf = X_test.copy()


prediction_pdf[
    "Actual_Is_Fraud"
] = y_test.values


prediction_pdf[
    "Predicted_Is_Fraud_Default"
] = y_pred_default


prediction_pdf[
    "Predicted_Is_Fraud_Tuned"
] = y_pred_tuned


prediction_pdf[
    "Fraud_Probability"
] = y_probability


print(
    "Prediction dataset shape:",
    prediction_pdf.shape
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 31. Drop Existing Prediction Table
# MAGIC
# MAGIC The existing table currently points to the Unity Catalog managed location:
# MAGIC
# MAGIC `s3://ujjivanpoc/Gold/__unitystorage/...`
# MAGIC
# MAGIC We are recreating it at:
# MAGIC
# MAGIC `s3://ujjivanpoc/Gold/fraud_predictions/`


# COMMAND ----------

table_exists = spark.catalog.tableExists(
    PREDICTION_TABLE
)

print(
    "Prediction table exists:",
    table_exists
)


# COMMAND ----------

if table_exists:

    print(
        f"Dropping existing table: "
        f"{PREDICTION_TABLE}"
    )

    spark.sql(
        f"DROP TABLE {PREDICTION_TABLE}"
    )

    print(
        "Existing prediction table dropped."
    )

else:

    print(
        "Prediction table does not exist."
    )


# COMMAND ----------
# MAGIC %md
# MAGIC ## 32. Write Predictions to User-Defined S3 Path


# COMMAND ----------

prediction_spark_df = (
    spark.createDataFrame(
        prediction_pdf
    )
)


(
    prediction_spark_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .option(
        "path",
        PREDICTION_S3_PATH
    )

    .saveAsTable(
        PREDICTION_TABLE
    )
)


print(
    "========================================"
)

print(
    "PREDICTION TABLE CREATED"
)

print(
    "========================================"
)

print(
    "Table:",
    PREDICTION_TABLE
)

print(
    "S3 path:",
    PREDICTION_S3_PATH
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 33. Verify Prediction Table Location


# COMMAND ----------

prediction_detail = spark.sql(
    f"DESCRIBE DETAIL {PREDICTION_TABLE}"
)


prediction_detail.select(
    "name",
    "format",
    "location"
).show(
    truncate=False
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 34. Verify Prediction Data


# COMMAND ----------

spark.table(
    PREDICTION_TABLE
).show(
    10,
    truncate=False
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 35. Create MLflow Signature


# COMMAND ----------

input_example = X_train.iloc[:5]

prediction_example = fraud_model.predict(
    input_example
)


signature = infer_signature(

    X_train,

    prediction_example

)


print(
    "MLflow signature created."
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 36. Start MLflow Run


# COMMAND ----------

with mlflow.start_run(
    run_name="bank_fraud_xgboost"
) as run:

    # ========================================
    # PARAMETERS
    # ========================================

    mlflow.log_param(
        "model_type",
        "XGBoost"
    )

    mlflow.log_param(
        "n_estimators",
        300
    )

    mlflow.log_param(
        "max_depth",
        6
    )

    mlflow.log_param(
        "learning_rate",
        0.05
    )

    mlflow.log_param(
        "subsample",
        0.8
    )

    mlflow.log_param(
        "colsample_bytree",
        0.8
    )

    mlflow.log_param(
        "scale_pos_weight",
        float(scale_pos_weight)
    )

    mlflow.log_param(
        "best_iteration",
        int(
            fraud_model.best_iteration
        )
    )

    mlflow.log_param(
        "target_recall",
        TARGET_RECALL
    )

    mlflow.log_param(
        "chosen_threshold",
        float(chosen_threshold)
    )


    # ========================================
    # METRICS
    # ========================================

    mlflow.log_metric(
        "roc_auc",
        float(roc_auc)
    )

    mlflow.log_metric(
        "pr_auc",
        float(pr_auc)
    )


    # ========================================
    # LOG + REGISTER MODEL
    # ========================================

    mlflow.xgboost.log_model(

        xgb_model=fraud_model,

        artifact_path="fraud_xgboost_model",

        signature=signature,

        input_example=input_example,

        registered_model_name=MODEL_NAME

    )


    run_id = run.info.run_id


    print(
        "========================================"
    )

    print(
        "MLFLOW MODEL REGISTRATION COMPLETED"
    )

    print(
        "========================================"
    )

    print(
        "Run ID:",
        run_id
    )

    print(
        "Model:",
        MODEL_NAME
    )


# COMMAND ----------
# MAGIC %md
# MAGIC ## 37. Verify Unity Catalog Model


# COMMAND ----------

from mlflow import MlflowClient


client = MlflowClient(
    registry_uri="databricks-uc"
)


registered_model = (
    client.get_registered_model(
        MODEL_NAME
    )
)


print(
    "========================================"
)

print(
    "REGISTERED MODEL"
)

print(
    "========================================"
)

print(
    "Name:",
    registered_model.name
)

print(
    "Description:",
    registered_model.description
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 38. List Model Versions


# COMMAND ----------

versions = list(
    client.search_model_versions(
        f"name = '{MODEL_NAME}'"
    )
)


if not versions:

    raise ValueError(
        f"No model versions found for "
        f"{MODEL_NAME}"
    )


print(
    "Model versions:"
)


for version in versions:

    print(
        "Version:",
        version.version,
        "| Run ID:",
        version.run_id
    )


# COMMAND ----------
# MAGIC %md
# MAGIC ## 39. Identify Latest Model Version


# COMMAND ----------

latest_version = max(
    versions,
    key=lambda x: int(x.version)
)


LATEST_MODEL_VERSION = (
    latest_version.version
)


print(
    "========================================"
)

print(
    "LATEST MODEL VERSION"
)

print(
    "========================================"
)

print(
    "Model:",
    MODEL_NAME
)

print(
    "Version:",
    LATEST_MODEL_VERSION
)

print(
    "Run ID:",
    latest_version.run_id
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 40. Load Registered Model


# COMMAND ----------

model_uri = (
    f"models:/{MODEL_NAME}/{LATEST_MODEL_VERSION}"
)


print(
    "Loading model:"
)

print(
    model_uri
)


loaded_model = mlflow.xgboost.load_model(
    model_uri
)


print(
    "Registered model loaded successfully."
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 41. Test Registered Model


# COMMAND ----------

test_predictions = (
    loaded_model.predict(
        X_test.iloc[:10]
    )
)


print(
    "Sample predictions:"
)

print(
    test_predictions
)


# COMMAND ----------
# MAGIC %md
# MAGIC ## 42. Final Verification


# COMMAND ----------

print("\n")
print("=" * 70)
print("BANK FRAUD XGBOOST — FINAL SUMMARY")
print("=" * 70)

print("\nSOURCE")
print(
    "  Table:",
    SOURCE_TABLE
)

print("\nPREDICTIONS")
print(
    "  Table:",
    PREDICTION_TABLE
)

print(
    "  S3:",
    PREDICTION_S3_PATH
)

print("\nMODEL")
print(
    "  Name:",
    MODEL_NAME
)

print(
    "  Version:",
    LATEST_MODEL_VERSION
)

print("\nMETRICS")
print(
    "  ROC-AUC:",
    roc_auc
)

print(
    "  PR-AUC:",
    pr_auc
)

print("\nTHRESHOLD")
print(
    "  Target Recall:",
    TARGET_RECALL
)

print(
    "  Chosen Threshold:",
    chosen_threshold
)

print("\nMLFLOW")
print(
    "  Run ID:",
    run_id
)

print("\n")
print("=" * 70)
print("NEW MODEL CREATED AND REGISTERED SUCCESSFULLY")
print("=" * 70)